# IMRCS-EM (K=10) on Expedia 10k — Single-Model Deep Dive

Focused analysis of **`record/10k_sample_expedia_imrcs_em_k10_unique.txt`** — the de-duplicated
`IMRCS_EM_K10` results (best run per `(site, trial)`).

**Design:** 5 sites × 10 trials = **50 instances**. Train/test ≈ 6000 / 4000 sessions per instance (a 10k sample split 60/40).

**What this notebook does** (a cross-model comparison already lives in `analyze_expedia_imrcs_comparison.ipynb`; here we drill into *this* model alone):
1. Load + integrity checks (which columns are constant, session counts, the χ²/weighted-sq relationship)
2. Overall summary statistics
3. **Per-site performance** (mean ± std over the 10 trials) — the core tables/plots
4. Train-vs-test fit (overfitting check, per session)
5. **EM convergence diagnostics** — `numIter` hits the cap of 30 on every run; what `finalImprove` tells us
6. Trial-to-trial stability (coefficient of variation)
7. Key findings

**Convention:** lower is better for NLL, RMSE, weighted-sq, χ²100, MAPE, time; higher is better for hit@1/2/3.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd()
# Run from repo root or from record/ — resolve either way.
RECORD = ROOT / 'record' if (ROOT / 'record').exists() else ROOT
DATA = RECORD / '10k_sample_expedia_imrcs_em_k10_unique.txt'
FIG_DIR = RECORD / 'expedia_imrcs_figures'
FIG_DIR.mkdir(exist_ok=True)

# Display order used throughout the project (paper site order).
SITE_ORDER = [15, 14, 5, 24, 32]

df = pd.read_csv(DATA, sep=r'\s+')
df['site'] = pd.Categorical(df['site'], categories=SITE_ORDER, ordered=True)
print('shape:', df.shape)
print('columns:', list(df.columns))
df.head()

## 1. Structure & integrity checks

Confirm the experiment grid (5 sites × 10 trials), see which columns are fixed knobs vs. measured
outcomes, and sanity-check two relationships in the file.

In [ ]:
# Experiment grid
grid = df.pivot_table(index='site', columns='trial', values='testNLL', aggfunc='size', observed=True)
print('Instances per (site x trial) — should be all 1s:')
print(grid.fillna(0).astype(int))
print('\nTotal instances:', len(df), '| sites:', sorted(df.site.unique().tolist()),
      '| trials:', sorted(df.trial.unique().tolist()))

# Which columns are constant across all 50 runs (fixed knobs)?
const_cols = {c: df[c].iloc[0] for c in df.columns if df[c].nunique() == 1}
print('\nConstant columns (fixed hyper-parameters / settings):')
for c, v in const_cols.items():
    print(f'  {c:>14} = {v}')

print('\nSession-count split:')
print('  trainSessions:', sorted(df.trainSessions.unique().tolist()),
      '| testSessions:', sorted(df.testSessions.unique().tolist()))

In [ ]:
# Relationship check: testChiSq100 vs testWeightedSq.
ratio = (df.testChiSq100 / df.testWeightedSq)
print(f'testChiSq100 / testWeightedSq : mean={ratio.mean():.4f}  std={ratio.std():.2e}')
print('-> the two are the same quantity at a x100 scale (independent rounding aside);')
print('   so we report χ²100 and drop weightedSq from plots to avoid redundancy.')
print(f'\nCorrelation testChiSq100 vs testWeightedSq: {df.testChiSq100.corr(df.testWeightedSq):.6f}')

## 2. Overall summary statistics

Distribution of every outcome metric across all 50 instances.

In [ ]:
METRICS = ['testNLL', 'testRMSE', 'testChiSq100', 'testMAPE',
           'hit1', 'hit2', 'hit3', 'purchaseRate', 'trainNLL', 'finalImprove', 'time']
overall = df[METRICS].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
overall['cv_%'] = 100 * df[METRICS].std() / df[METRICS].mean()
overall.round(4)

## 3. Per-site performance (the core)

Each site is averaged over its 10 trials. `mean ± std` for every metric, in paper site order.
This is the table that summarizes how the model behaves across markets.

In [ ]:
agg_metrics = ['testNLL', 'testRMSE', 'testChiSq100', 'testMAPE',
               'hit1', 'hit2', 'hit3', 'purchaseRate', 'finalImprove', 'time']
per_site = df.groupby('site', observed=True)[agg_metrics].agg(['mean', 'std'])
# Flatten to "metric_mean / metric_std" and present mean (std) compactly.
per_site_compact = pd.DataFrame(index=SITE_ORDER)
for m in agg_metrics:
    mu, sd = per_site[(m, 'mean')], per_site[(m, 'std')]
    per_site_compact[m] = [f'{mu[s]:.4f} ({sd[s]:.4f})' for s in SITE_ORDER]
per_site_compact.index.name = 'site'

# Save the numeric version for downstream use (paper tables etc.)
per_site_numeric = per_site.copy()
per_site_numeric.columns = [f'{m}_{stat}' for m, stat in per_site_numeric.columns]
out_csv = RECORD / 'imrcs_em_k10_unique_per_site.csv'
per_site_numeric.round(6).to_csv(out_csv)
print('saved per-site numeric summary ->', out_csv.name)
per_site_compact

In [ ]:
# Best / worst site per metric (over trial means).
site_means = df.groupby('site', observed=True)[agg_metrics].mean()
LOWER_BETTER = {'testNLL', 'testRMSE', 'testChiSq100', 'testMAPE', 'finalImprove', 'time'}
rows = []
for m in agg_metrics:
    asc = m in LOWER_BETTER
    order = site_means[m].sort_values(ascending=asc)
    rows.append({'metric': m, 'direction': 'lower=better' if asc else 'higher=better',
                 'best_site': int(order.index[0]), 'best_val': round(order.iloc[0], 4),
                 'worst_site': int(order.index[-1]), 'worst_val': round(order.iloc[-1], 4)})
pd.DataFrame(rows).set_index('metric')

### 3a. Per-site fit metrics (error bars = ±1 std over trials)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
x = np.arange(len(SITE_ORDER))
for ax, m, title in zip(axes,
                        ['testRMSE', 'testChiSq100', 'testMAPE'],
                        ['test RMSE', 'test χ²100', 'test MAPE']):
    mu = [site_means.loc[s, m] for s in SITE_ORDER]
    sd = [df[df.site == s][m].std() for s in SITE_ORDER]
    bars = ax.bar(x, mu, yerr=sd, capsize=4, color='#4C72B0', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(SITE_ORDER)
    ax.set_title(title + ' by site'); ax.set_xlabel('site')
    best = int(np.argmin(mu)); bars[best].set_color('#55A868')  # green = best (lower)
fig.suptitle('Per-site fit (green = best, lower is better)', y=1.04, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_per_site_fit.png', bbox_inches='tight')
plt.show()

### 3b. Per-site hit rates (higher is better)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.26
for i, (m, c) in enumerate(zip(['hit1', 'hit2', 'hit3'], ['#C44E52', '#DD8452', '#8C8C8C'])):
    mu = [site_means.loc[s, m] for s in SITE_ORDER]
    sd = [df[df.site == s][m].std() for s in SITE_ORDER]
    ax.bar(x + (i - 1) * w, mu, w, yerr=sd, capsize=3, label=m, color=c, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(SITE_ORDER)
ax.set_xlabel('site'); ax.set_ylabel('hit rate'); ax.set_title('Hit@1/2/3 by site')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_per_site_hits.png', bbox_inches='tight')
plt.show()

### 3c. Trial-level spread per site

Box = IQR across the 10 trials; dots = individual trials. Shows how much the random restart / data split moves each metric.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m, title in zip(axes, ['testRMSE', 'testNLL'], ['test RMSE', 'test NLL']):
    data = [df[df.site == s][m].values for s in SITE_ORDER]
    ax.boxplot(data, labels=SITE_ORDER, showmeans=True, widths=0.6)
    for i, s in enumerate(SITE_ORDER, start=1):
        y = df[df.site == s][m].values
        ax.scatter(np.full_like(y, i) + np.linspace(-0.12, 0.12, len(y)), y,
                   s=18, alpha=0.6, color='#4C72B0', zorder=3)
    ax.set_title(f'{title} across 10 trials, by site'); ax.set_xlabel('site')
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_trial_spread.png', bbox_inches='tight')
plt.show()

## 4. Train vs. test fit (overfitting check)

`trainNLL` is summed over ~6000 sessions, `testNLL` over ~4000, so raw values aren't comparable.
Put both on a **per-session** footing and compare. If per-session test NLL >> train NLL, the K=10
mixture is overfitting.

In [ ]:
df = df.assign(
    trainNLL_persession=df.trainNLL / df.trainSessions,
    testNLL_persession=df.testNLL / df.testSessions,
)
df['nll_gap'] = df.testNLL_persession - df.trainNLL_persession

gap_tbl = df.groupby('site', observed=True)[['trainNLL_persession', 'testNLL_persession', 'nll_gap']].mean()
print('Per-session NLL (mean over trials):')
print(gap_tbl.reindex(SITE_ORDER).round(4))

fig, ax = plt.subplots(figsize=(5.5, 5))
colors = dict(zip(SITE_ORDER, plt.cm.tab10.colors))
for s in SITE_ORDER:
    sub = df[df.site == s]
    ax.scatter(sub.trainNLL_persession, sub.testNLL_persession,
               label=f'site {s}', color=colors[s], s=40, alpha=0.85)
lo = min(df.trainNLL_persession.min(), df.testNLL_persession.min())
hi = max(df.trainNLL_persession.max(), df.testNLL_persession.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='train = test')
ax.set_xlabel('train NLL / session'); ax.set_ylabel('test NLL / session')
ax.set_title('Per-session train vs test NLL\n(points above line = generalization gap)')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_train_vs_test.png', bbox_inches='tight')
plt.show()

## 5. EM convergence diagnostics

`numIter = 30` on **every** run and `numPatience = 4` — the EM loop is hitting its **iteration cap**,
not an early-stop on convergence. The telling quantity is `finalImprove`: the objective gain at the
*last* iteration. If that were ~0 the model had converged; here it ranges 1.8–4.6, so EM was **still
improving when it was cut off**. This is a knob worth revisiting (raise the iteration cap) — does the
remaining improvement track worse test fit?

In [ ]:
print('numIter unique     :', df.numIter.unique(), '  (cap reached on every run)')
print('numPatience unique :', df.numPatience.unique())
print('finalImprove       : min={:.3f}  median={:.3f}  max={:.3f}'.format(
      df.finalImprove.min(), df.finalImprove.median(), df.finalImprove.max()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# (a) finalImprove distribution by site
data = [df[df.site == s].finalImprove.values for s in SITE_ORDER]
axes[0].boxplot(data, labels=SITE_ORDER, showmeans=True, widths=0.6)
axes[0].set_title('finalImprove (last-iter EM gain) by site\nhigher = further from convergence at cap')
axes[0].set_xlabel('site'); axes[0].set_ylabel('finalImprove')

# (b) does remaining improvement correlate with test fit?
axes[1].scatter(df.finalImprove, df.testNLL_persession, c=[colors[s] for s in df.site], s=40)
r = df.finalImprove.corr(df.testNLL_persession)
axes[1].set_xlabel('finalImprove (gain at last EM iter)')
axes[1].set_ylabel('test NLL / session')
axes[1].set_title(f'Remaining improvement vs test fit\nPearson r = {r:.3f}')
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_em_diag.png', bbox_inches='tight')
plt.show()

## 6. Trial-to-trial stability

Coefficient of variation (CV = std / mean, %) across the 10 trials, per site. Small CV = the result is
robust to the random restart and data split; large CV = sensitive. Which market is least stable?

In [ ]:
cv_metrics = ['testNLL', 'testRMSE', 'testChiSq100', 'testMAPE', 'hit1', 'hit2', 'hit3', 'finalImprove']
cv = df.groupby('site', observed=True)[cv_metrics].apply(lambda g: 100 * g.std() / g.mean())
cv = cv.reindex(SITE_ORDER)
cv.index.name = 'site (CV % over 10 trials)'

fig, ax = plt.subplots(figsize=(9, 3.6))
im = ax.imshow(cv.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(cv_metrics))); ax.set_xticklabels(cv_metrics, rotation=35, ha='right')
ax.set_yticks(range(len(SITE_ORDER))); ax.set_yticklabels([f'site {s}' for s in SITE_ORDER])
for i in range(cv.shape[0]):
    for j in range(cv.shape[1]):
        ax.text(j, i, f'{cv.values[i, j]:.1f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='CV %')
ax.set_title('Trial-to-trial variability (CV %) — darker = noisier')
fig.tight_layout()
fig.savefig(FIG_DIR / 'k10unique_stability.png', bbox_inches='tight')
plt.show()
cv.round(2)

## 7. Key findings

*(Generated from the cells above — numbers refer to this run.)*

In [ ]:
sm = df.groupby('site', observed=True)[agg_metrics].mean()
best_rmse = sm.testRMSE.idxmin(); worst_rmse = sm.testRMSE.idxmax()
best_nll  = sm.testNLL.idxmin();  worst_nll  = sm.testNLL.idxmax()
print('KEY FINDINGS')
print('=' * 64)
print(f'1. Best-fit market  : site {best_rmse} (RMSE {sm.testRMSE[best_rmse]:.4f}, '
      f'NLL {sm.testNLL[best_nll]:.0f}); also the lowest purchaseRate '
      f'({sm.purchaseRate[best_rmse]:.3f}).')
print(f'2. Worst-fit market : site {worst_rmse} (RMSE {sm.testRMSE[worst_rmse]:.4f}, '
      f'χ²100 {sm.testChiSq100[worst_rmse]:.3f}) — worst on RMSE, χ²100 and MAPE.')
print(f'3. EM did NOT converge: numIter==30 on all 50 runs (the cap); '
      f'finalImprove still {df.finalImprove.min():.2f}-{df.finalImprove.max():.2f}.')
print(f'   -> Raising the iteration cap is the obvious next experiment.')
gap = df.nll_gap.mean()
print(f'4. Generalization gap: mean per-session (test-train) NLL = {gap:+.4f} '
      f'nats/session ({"test worse" if gap>0 else "no overfit"}).')
print(f'5. Stability: hit@1 is the noisiest metric '
      f'(CV up to {cv.hit1.max():.1f}%); fit metrics (RMSE/χ²/NLL) are tight (CV < {cv[["testRMSE","testNLL","testChiSq100"]].max().max():.1f}%).')
print(f'6. Runtime: {df.time.mean():.2f}s/run on average (range {df.time.min():.2f}-{df.time.max():.2f}s).')
print('=' * 64)